# Compute Forward Operator

## 📦 Imports

In [1]:
import mindmap as mm 
import mne
import pyvista as pv

## 🧠 Data

📥 **<font color=pink>Inputs</font>**
- `bem`: Boundary Element Model (Head conductor model).
- `src-raw`: Source space (before computing forward).
- `eeg`: EEG.
- `trans`: Affine transformation for EEG coregistration.

🎚️ **<font color=pink>Configuration</font>**:
- `fconfig`: Configuration file.
- `sub`: Subject id (position in list SUBS)
- `hmodel`: Type of head model ["bem", "bem-sim"]. Whether to compute the forward solution for inverse problems or simulations.

In [2]:
# ------------ Parameters -------------
fconfig = "../../papers/intra/configuration.yml"
sub     = "12"
hmodel  = "bem"
task    = "spk"
# -------------------------------------

reader = mm.io.dataset_reader(fconfig)
mm_dir = reader.config["MM_DIR"]

# data
data = reader(sub, hmodel, "src-raw", "ave", "trans", task=task)

DATA SUMMARY
Dataset: intra
Subject: 12
Task: spk
Run: None
Retrieved objects:
- trans
- bem
- src-raw
- ave
Missing objects:


📊 **Check coregistration**

In [ ]:
head = data[hmodel]["surfs"][0]
sens = data["ave"]._get_channel_positions()
sens = mne.transforms.apply_trans(data["trans"], sens)

# plot
head = mm.head.surf_to_polydata(head["rr"], head["tris"])
sens = pv.PolyData(sens)

p = pv.Plotter()
p.add_mesh(head, color="#D2A18C", smooth_shading=False, label="head")
p.add_mesh(sens, render_points_as_spheres=True, color="black", label="sensors", point_size=8)

p.add_legend(face="circle")
p.show()

Widget(value='<iframe src="http://localhost:43417/index.html?ui=P_0x7af161661a50_0&reconnect=auto" class="pyvi…

/home/nino/miniconda3/envs/mindmap/lib/python3.11/site-packages/pyvista/core/utilities/misc.py:223: UserWarning: Encountered issue in callback (most recent call last):
  File "/tmp/ipykernel_6835/3175911736.py", line 44, in pick_electrode
    mesh_params={"clim":clim}
                        ^^^^
NameError: name 'clim' is not defined
  warnings.warn(formatted_exception)


{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color': None}
{'smooth_shading': True, 'clim': [np.float32(-1381.0499), np.float32(1381.0499)], 'cmap': 'bwr', 'color'

## ⚙️ Compute Forward Solution

In [4]:
forward = mne.make_forward_solution(
    info    = data["ave"].info,
    trans   = data["trans"],
    src     = data["src-raw"],
    bem     = data[hmodel],
    meg     = False,
    eeg     = True,
    verbose = "warning",
)

In [15]:
if reader.config.get("TASKS"):
    print("yes")
else:
    print("no")

yes


📊 **Plot leadfields**

In [10]:
mm.viz.plot_leadfield(data["ave"], forward).show()

    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]


TypeError: plot_brain() got an unexpected keyword argument 'clim'

💾 **Save Forward Solution** <br>
🎚️ **<font color=pink>Save</font>**:
- `overwrite`: Overwrite existing forward solution.


In [ ]:
# ------------ Parameters -------------
overwrite = True
# -------------------------------------

if hmodel == "bem":
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_desc-inverse_fwd.fif"
else:
    ffwd = mm_dir/data["sub"]/"fwd"/f"{data['sub']}_desc-simulation_fwd.fif"

ffwd.parent.mkdir(parents=True, exists_ok=True)

mne.write_forward_solution(ffwd, forward, overwrite=overwrite, verbose="warning")
print(f"\033[92mSuccessfully saved forward solution for subject {data['sub']} with model '{hmodel}'\033[0m")

Successfully saved forward solution for subject vepcon_sub-10 with model 'bem'
